In [0]:
dbutils.widgets.text('catalog', 'dev')
catalog = dbutils.widgets.get('catalog')

In [0]:
# Build fact_sales Gold table

import pyspark.sql.functions as F
from pyspark.sql import DataFrame

# Load Silver tables as DataFrames
lineitem = spark.table(f'{catalog}.silver.lineitem')
orders = spark.table(f'{catalog}.silver.orders')
customer = spark.table(f'{catalog}.silver.customer')
part = spark.table(f'{catalog}.silver.part')
supplier = spark.table(f'{catalog}.silver.supplier')
nation = spark.table(f'{catalog}.silver.nation')
region = spark.table(f'{catalog}.silver.region')

# Join with orders, customer, part, supplier, nation and region for dimension keys
fact_sales = (
    lineitem
    .join(orders, lineitem.l_orderkey == orders.o_orderkey, 'left')
    .join(customer, orders.o_custkey == customer.c_custkey, 'left')
    .join(part, lineitem.l_partkey == part.p_partkey, 'left')
    .join(supplier, lineitem.l_suppkey == supplier.s_suppkey, 'left')
    .join(nation, customer.c_nationkey == nation.n_nationkey, 'left')
    .join(region, nation.n_regionkey == region.r_regionkey, 'left')
    .select(
        # Core keys
        lineitem.l_orderkey.alias('orderkey'),
        lineitem.l_linenumber.alias('linenumber'),
        customer.c_custkey.alias('customerkey'),
        part.p_partkey.alias('productkey'),
        supplier.s_suppkey.alias('supplierkey'),
        nation.n_nationkey.alias('nationkey'),
        region.r_regionkey.alias('regionkey'),
        orders.o_orderdate.alias('orderdate'),
        # Measures (use business metrics)
        F.col('l_extendedprice').alias('extendedprice'),
        F.col('l_discount').alias('discount'),
        (F.col('l_extendedprice') * (1 - F.col('l_discount'))).alias('net_revenue'),  # Metric definition
        F.col('l_quantity').alias('quantity'),
        F.col('l_tax').alias('tax'),
        # Attributes
        F.col('l_shipdate'),
        F.col('l_commitdate'),
        F.col('l_receiptdate'),
        F.col('l_returnflag'),
        F.col('l_linestatus'),
        F.col('o_orderstatus'),
        F.col('o_orderpriority'),
        F.col('o_clerk'),
        F.col('o_shippriority'),
        F.col('c_name').alias('customer_name'),
        F.col('p_name').alias('product_name'),
        F.col('s_name').alias('supplier_name'),
        F.col('n_name').alias('nation_name'),
        F.col('r_name').alias('region_name')
    )
)

# Save as Gold table
fact_sales.write.mode('overwrite').format('delta').saveAsTable(f'{catalog}.gold.fact_sales')

print(f'Gold fact_sales table written: {catalog}.gold.fact_sales')

In [0]:
# Build dim_customer Gold table

import pyspark.sql.functions as F
from pyspark.sql import DataFrame

customer = spark.table(f'{catalog}.silver.customer')

# Select columns with business names and validate key uniqueness
dim_customer = (
    customer
    .select(
        F.col('c_custkey').alias('customerkey'),
        F.col('c_name').alias('customer_name'),
        F.col('c_address').alias('address'),
        F.col('c_nationkey').alias('nationkey'),
        F.col('c_phone').alias('phone'),
        F.col('c_acctbal').alias('account_balance'),
        F.col('c_mktsegment').alias('market_segment'),
        F.col('c_comment').alias('comment')
    )
)

# Validate uniqueness of the customer key
unique_count = dim_customer.select('customerkey').distinct().count()
total_rows = dim_customer.count()
if unique_count == total_rows:
    print('PASS: dim_customer key is unique for all rows')
else:
    print(f'FAIL: dim_customer key uniqueness violated ({unique_count} unique keys, {total_rows} rows)')

# Save as Gold table
dim_customer.write.mode('overwrite').format('delta').saveAsTable(f'{catalog}.gold.dim_customer')

print(f'Gold dim_customer table written: {catalog}.gold.dim_customer')

In [0]:
# Build dim_product Gold Table

import pyspark.sql.functions as F
from pyspark.sql import DataFrame

product = spark.table(f'{catalog}.silver.part')

dim_product = (
    product
    .select(
        F.col('p_partkey').alias('productkey'),
        F.col('p_name').alias('product_name'),
        F.col('p_mfgr').alias('manufacturer'),
        F.col('p_brand').alias('brand'),
        F.col('p_type').alias('type'),
        F.col('p_size').alias('size'),
        F.col('p_container').alias('container'),
        F.col('p_retailprice').alias('retail_price'),
        F.col('p_comment').alias('comment')
    )
)

unique_count = dim_product.select('productkey').distinct().count()
total_rows = dim_product.count()
if unique_count == total_rows:
    print('PASS: dim_product key is unique for all rows')
else:
    print(f'FAIL: dim_product key uniqueness violated ({unique_count} unique keys, {total_rows} rows)')

dim_product.write.mode('overwrite').format('delta').saveAsTable(f'{catalog}.gold.dim_product')

print(f'Gold dim_product table written: {catalog}.gold.dim_product')

In [0]:
# Build dim_supplier Gold Table

import pyspark.sql.functions as F
from pyspark.sql import DataFrame

supplier = spark.table(f'{catalog}.silver.supplier')

dim_supplier = (
    supplier
    .select(
        F.col('s_suppkey').alias('supplierkey'),
        F.col('s_name').alias('supplier_name'),
        F.col('s_address').alias('address'),
        F.col('s_nationkey').alias('nationkey'),
        F.col('s_phone').alias('phone'),
        F.col('s_acctbal').alias('account_balance'),
        F.col('s_comment').alias('comment')
    )
)

unique_count = dim_supplier.select('supplierkey').distinct().count()
total_rows = dim_supplier.count()
if unique_count == total_rows:
    print('PASS: dim_supplier key is unique for all rows')
else:
    print(f'FAIL: dim_supplier key uniqueness violated ({unique_count} unique keys, {total_rows} rows)')

dim_supplier.write.mode('overwrite').format('delta').saveAsTable(f'{catalog}.gold.dim_supplier')

print(f'Gold dim_supplier table written: {catalog}.gold.dim_supplier')

In [0]:
# Build dim_geography Gold Table

import pyspark.sql.functions as F
from pyspark.sql import DataFrame

nation = spark.table(f'{catalog}.silver.nation')
region = spark.table(f'{catalog}.silver.region')

dim_geography = (
    nation
    .join(region, nation.n_regionkey == region.r_regionkey, 'left')
    .select(
        F.col('n_nationkey').alias('nationkey'),
        F.col('n_name').alias('nation_name'),
        F.col('r_regionkey').alias('regionkey'),
        F.col('r_name').alias('region_name'),
        F.col('n_comment').alias('nation_comment'),
        F.col('r_comment').alias('region_comment')
    )
)

unique_count = dim_geography.select('nationkey').distinct().count()
total_rows = dim_geography.count()
if unique_count == total_rows:
    print('PASS: dim_geography nationkey is unique for all rows')
else:
    print(f'FAIL: dim_geography nationkey uniqueness violated ({unique_count} unique keys, {total_rows} rows)')

dim_geography.write.mode('overwrite').format('delta').saveAsTable(f'{catalog}.gold.dim_geography')

print(f'Gold dim_geography table written: {catalog}.gold.dim_geography')

In [0]:
# Build vw_annual_sales_kpis Gold View

import pyspark.sql.functions as F

fact_sales = spark.table(f'{catalog}.gold.fact_sales')

# Derive year from orderdate for annual metrics
vw_annual_sales_kpis = (
    fact_sales
    .groupBy(F.year('orderdate').alias('order_year'))
    .agg(
        F.sum('net_revenue').alias('annual_net_revenue'),
        F.countDistinct('orderkey').alias('annual_order_count'),
        F.countDistinct('customerkey').alias('annual_active_customers'),
        (F.sum('net_revenue') / F.countDistinct('orderkey')).alias('average_order_value')
    )
    .orderBy('order_year')
)

vw_annual_sales_kpis.write.mode('overwrite').format('delta').saveAsTable(f'{catalog}.gold.vw_annual_sales_kpis')

print(f'Gold annual KPIs view written: {catalog}.gold.vw_annual_sales_kpis')

In [0]:
# Reconciliation: Gold vs Silver

import pyspark.sql.functions as F

# --- Silver layer totals (source of truth) ---
silver_lineitem = spark.table(f'{catalog}.silver.lineitem')
silver_orders = spark.table(f'{catalog}.silver.orders')

silver_row_count = silver_lineitem.count()
silver_net_revenue = silver_lineitem.agg(F.sum(F.col('l_extendedprice') * (1 - F.col('l_discount')))).collect()[0][0]
silver_order_count = silver_lineitem.select(F.countDistinct('l_orderkey')).collect()[0][0]
silver_customer_count = silver_orders.select(F.countDistinct('o_custkey')).collect()[0][0]

# --- Gold layer totals ---
gold_fact = spark.table(f'{catalog}.gold.fact_sales')

gold_row_count = gold_fact.count()
gold_net_revenue = gold_fact.agg(F.sum('net_revenue')).collect()[0][0]
gold_order_count = gold_fact.select(F.countDistinct('orderkey')).collect()[0][0]
gold_customer_count = gold_fact.select(F.countDistinct('customerkey')).collect()[0][0]

# --- Compare ---
print('=== Gold vs Silver Reconciliation ===')
print()

checks = [
    ('Row count', silver_row_count, gold_row_count),
    ('Net revenue', silver_net_revenue, gold_net_revenue),
    ('Distinct orders', silver_order_count, gold_order_count),
    ('Distinct customers', silver_customer_count, gold_customer_count),
]

all_pass = True
for label, silver_val, gold_val in checks:
    match = (silver_val == gold_val)
    status = 'PASS' if match else 'FAIL'
    if not match:
        all_pass = False
    print(f'{status}: {label} — Silver: {silver_val}  |  Gold: {gold_val}')

print()
if all_pass:
    print('ALL RECONCILIATION CHECKS PASSED')
else:
    print('RECONCILIATION FAILED — investigate mismatches above')